# NB-R06 — Regime-Stratified Baseline Comparison

**Pipeline stage:** 6 of 13

**Purpose.** Establish whether the stacking ensemble's complexity is justified by comparing it, under the identical High-VIX/Low-VIX split, against simple baselines: a majority-class rule, a naive persistence rule, logistic regression, and a single (untuned-stack) XGBoost model.

**Why this matters.** A high accuracy number within a small, class-imbalanced subset (the High-VIX regime has only 8 observations, all realizing the same outcome) can look impressive without reflecting any real skill above a trivial rule. Reporting the same regime split's majority-class baseline alongside every model's accuracy makes that distinction explicit.

**Inputs:** `data/processed/test_with_regimes.csv`, trained base models from NB-R03.

**Outputs:** `results/baseline_comparison.csv`, `plots/R06_baseline_comparison.png`.

**Result:** on the High-VIX subset every model that predicts "Up" matches the 100% majority baseline trivially (n=8, single class). On the Low-VIX subset, the stacking ensemble (66.5%) beats every individual baseline tested but still falls short of that subset's own 70.2% majority baseline.


In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import xgboost as xgb

PROJ    = Path('..').resolve()  # repo root, assuming this notebook is run from notebooks/
PROC    = PROJ / 'data' / 'processed'
RESULTS = PROJ / 'results'
PLOTS   = PROJ / 'plots'
MODELS  = PROJ / 'models'

with open(PROC / 'feature_cols.json') as f:
    FEATURE_COLS = json.load(f)
with open(RESULTS / 'all_best_params.json') as f:
    best_params = json.load(f)

SEED = 42

train = pd.read_csv(PROC / 'train.csv', parse_dates=['date'])
val   = pd.read_csv(PROC / 'val.csv',   parse_dates=['date'])
test  = pd.read_csv(PROC / 'test_predictions.csv', parse_dates=['date'])

X_train = train[FEATURE_COLS].values
y_train = train['dir_21d'].values
X_test  = test[FEATURE_COLS].values
y_test  = test['dir_21d'].values

regime = test['regime_fixed'].values
print(f'Train: {len(train)}, Test: {len(test)}')

## 1. Train Baseline Models

In [ ]:
# Logistic Regression
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]
lr_preds = lr.predict(X_test)
print('Logistic Regression trained.')

# Single XGBoost (best params)
xgb_p = {k: v for k, v in best_params['xgb'].items()}
xgb_single = xgb.XGBClassifier(**xgb_p)
xgb_single.fit(X_train, y_train, verbose=False)
xgb_probs = xgb_single.predict_proba(X_test)[:, 1]
xgb_preds = xgb_single.predict(X_test)
print('Single XGBoost trained.')

# Stacking ensemble predictions (from NB-R03)
stack_preds = test['stack_pred'].values
stack_probs = test['stack_prob'].values
print('Stacking ensemble predictions loaded.')

## 2. Regime-Stratified Comparison

In [ ]:
def regime_metrics(y_true, y_pred, y_prob, regime, model_name):
    rows = []
    for reg in ['High-VIX', 'Low-VIX', 'Overall']:
        if reg == 'Overall':
            mask = np.ones(len(y_true), dtype=bool)
        else:
            mask = regime == reg

        yt = y_true[mask]
        yp = y_pred[mask]
        ypr = y_prob[mask]

        maj_acc = max(yt.mean(), 1 - yt.mean()) * 100
        acc     = accuracy_score(yt, yp) * 100
        f1      = f1_score(yt, yp, zero_division=0) * 100
        auc     = roc_auc_score(yt, ypr) if len(np.unique(yt)) > 1 else np.nan

        rows.append({
            'Model': model_name, 'Regime': reg, 'N': int(mask.sum()),
            'Majority Baseline (%)': round(maj_acc, 1),
            'Accuracy (%)': round(acc, 1),
            'F1 (%)': round(f1, 1),
            'ROC-AUC': round(auc, 4) if not np.isnan(auc) else 'N/A'
        })
    return rows

# Majority-class baseline: always predict 1 (Up)
maj_preds = np.ones_like(y_test)
maj_probs = np.full(len(y_test), y_train.mean())   # calibrated to train prior

# Persistence baseline: predict dir_21d = sign of today's log return
# We use log_ret_lag1 as a proxy for "today's direction"
lag1_idx = FEATURE_COLS.index('log_ret_lag1')
persist_preds = (X_test[:, lag1_idx] > 0).astype(int)
persist_probs = (X_test[:, lag1_idx] + 0.5).clip(0, 1)   # soft prob proxy

all_rows = []
for name, pred, prob in [
    ('Majority-Class',   maj_preds,     maj_probs),
    ('Persistence',      persist_preds, persist_probs),
    ('Logistic Reg.',    lr_preds,      lr_probs),
    ('XGBoost (single)', xgb_preds,     xgb_probs),
    ('Stacking Ensemble',stack_preds,   stack_probs),
]:
    all_rows.extend(regime_metrics(y_test, pred, prob, regime, name))

results_df = pd.DataFrame(all_rows)
print(results_df.to_string(index=False))
results_df.to_csv(RESULTS / 'baseline_comparison.csv', index=False)

In [ ]:
# Plot: High-VIX accuracy comparison across models
high_vix_data = results_df[results_df['Regime'] == 'High-VIX'][['Model', 'Accuracy (%)']]
low_vix_data  = results_df[results_df['Regime'] == 'Low-VIX'][['Model', 'Accuracy (%)']]

fig, ax = plt.subplots(figsize=(10, 5))
models = high_vix_data['Model'].tolist()
x = np.arange(len(models))
ax.bar(x - 0.2, high_vix_data['Accuracy (%)'], width=0.35, label='High-VIX', color='tomato', alpha=0.85)
ax.bar(x + 0.2, low_vix_data['Accuracy (%)'],  width=0.35, label='Low-VIX',  color='steelblue', alpha=0.85)
ax.axhline(50, color='black', linestyle='--', linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Baseline Comparison: Regime-Stratified Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS / 'R06_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

---
## Summary

**Pipeline stage:** 6 of 13 (see `notebooks/README.md` for the full pipeline map).

**Artifacts produced by this notebook:**

- `results/baseline_comparison.csv`
- `plots/R06_baseline_comparison.png`

**Next notebook:** `NB-R07_shap_regime_analysis.ipynb`
